In [ ]:
# === Imports & EOS-80 density helper =========================================
import numpy as np
import pandas as pd
from pathlib import Path
import zipfile, tempfile, os
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import cm, colors
from matplotlib.lines import Line2D
import tfv.xarray  # registers the .tfv xarray accessor

def eos80_potential_density(S, T):
    """UNESCO EOS-80 seawater density referenced to the surface (p=0),
    i.e. potential density. Vectorised; accepts xarray/numpy. S in psu, T in degC."""
    T2, T3, T4, T5 = T*T, T*T*T, T*T*T*T, T*T*T*T*T
    Ssq = np.sqrt(np.clip(S, 0, None)); S1p5 = S*Ssq; S2 = S*S
    a = [999.842594, 6.793952e-2, -9.095290e-3, 1.001685e-4, -1.120083e-6, 6.536332e-9]
    rho_w = a[0] + a[1]*T + a[2]*T2 + a[3]*T3 + a[4]*T4 + a[5]*T5
    b = [8.24493e-1, -4.0899e-3, 7.6438e-5, -8.2467e-7, 5.3875e-9]
    c = [-5.72466e-3, 1.0227e-4, -1.6546e-6]; d0 = 4.8314e-4
    return (rho_w + (b[0]+b[1]*T+b[2]*T2+b[3]*T3+b[4]*T4)*S
            + (c[0]+c[1]*T+c[2]*T2)*S1p5 + d0*S2)

In [ ]:
# === Configuration ===========================================================
MODEL_NC  = Path(r'S:/Matt_Working/csiem/output_archive/1.7.0/1991_aug/csiem_B010_19910720_19910831.nc')
SHAPE_ZIP = Path(r'G:/CSIEM/1.8.0/csiem-marvl/custom_py/dadamo_transect/shapefile.zip')
OUT_DIR   = Path(r'G:/CSIEM/1.8.0/csiem-marvl/custom_py/dadamo_transect/outputs_1991')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Snapshot cadence: one figure per day, nearest to this hour (model is 4-hourly)
SNAPSHOT_HOUR = 12

# Set True to render just the first day (all 3 vars) inline for a quick check.
TEST_MODE = True

# Variables to plot and their display specs
VARIABLES = ['RHOW', 'SAL', 'TEMP']
VARSPECS = {
    'RHOW': dict(clim=(1024, 1027),  cmap='jet',      label='Water Density (kg m$^{-3}$)'),
    'SAL' : dict(clim=(34.8, 35.6),  cmap='viridis',  label='Salinity (psu)'),
    'TEMP': dict(clim=(15.0, 18.5),  cmap='RdYlBu_r', label='Temperature ($^\circ$C)'),
}
DEPTH_YLIM = (-22, 0)

# Transect stations (lon, lat) along the Cockburn Sound polyline
station_points = {
    'MR': (115.72983, -32.066497),
    'OA9S': (115.731672, -32.104168),
    'OA1-DEP': (115.726209, -32.124373),
    '6147030': (115.70274, -32.156587),
    '6142974': (115.719833, -32.197565),
    'LP': (115.746666, -32.226664),
    'SF11': (115.721449, -32.242053),
    '6142983': (115.712531, -32.248232),
    'SC': (115.684285, -32.254165),
    'SC2': (115.686273, -32.298863),
}
# Manual chainage (m from north) for station markers on the curtain
manual_station_chainage = {
    'MR': 600, 'OA9S': 6000, 'OA1-DEP': 10500, '6147030': 14500, '6142974': 19500,
    'LP': 26000, 'SF11': 31500, '6142983': 32900, 'SC': 36000, 'SC2': 44000,
}
sites = list(station_points.keys())

In [ ]:
# === Load transect polyline from the bundled shapefile =======================
_tmp = tempfile.mkdtemp()
zipfile.ZipFile(SHAPE_ZIP).extractall(_tmp)
gdf = gpd.read_file(os.path.join(_tmp, 'shapefile', 'polyline_1.shp'))
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

def geometry_to_coords(geom):
    if geom is None or geom.is_empty:
        return []
    gt = geom.geom_type
    if gt == 'Point':
        return [(geom.x, geom.y)]
    if gt in ('LineString', 'LinearRing'):
        return list(geom.coords)
    if gt == 'Polygon':
        return list(geom.exterior.coords)
    if gt.startswith('Multi') or gt == 'GeometryCollection':
        out = []
        for part in geom.geoms:
            out.extend(geometry_to_coords(part))
        return out
    return []

coords = []
for geom in gdf.geometry:
    coords.extend(geometry_to_coords(geom))
polyline = np.array(coords)
polyline = polyline[::-1].copy()   # orient south->north so chainage-from-north aligns
print(f'polyline: {len(polyline)} pts | lon[{polyline[:,0].min():.4f},{polyline[:,0].max():.4f}] '
      f'lat[{polyline[:,1].min():.4f},{polyline[:,1].max():.4f}]')

In [ ]:
# === Open model output and inject derived density (RHOW) =====================
ds = xr.open_dataset(MODEL_NC)
ds['RHOW'] = eos80_potential_density(ds['SAL'], ds['TEMP'])
ds['RHOW'].attrs.update(long_name='Potential density (EOS-80, p=0)', units='kg/m3')
fv = ds.tfv

times = pd.to_datetime(ds['Time'].values)
_days = pd.date_range(times.min().normalize(), times.max().normalize(), freq='D') \
        + pd.Timedelta(hours=SNAPSHOT_HOUR)
snapshot_dates = [d for d in _days if times.min() <= d <= times.max()]
print(f'Model coverage: {times.min()} -> {times.max()}  ({len(times)} steps)')
print(f'{len(snapshot_dates)} daily snapshots @ {SNAPSHOT_HOUR}:00 '
      f'({snapshot_dates[0].date()} -> {snapshot_dates[-1].date()})')

In [ ]:
# === Figure builder (model-only): curtain + station profiles + map inset =====
def make_transect_figure(var, model_date, save=True, show=False):
    spec = VARSPECS[var]
    clim, cmap, clabel = spec['clim'], spec['cmap'], spec['label']
    y_min, y_max = DEPTH_YLIM

    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(3, 6, width_ratios=[1, 1, 1, 1, 1, 0.95],
                          height_ratios=[1.2, 1, 1], hspace=0.25, wspace=0.10)
    ax_transect = fig.add_subplot(gs[0, :5])
    profile_axes = []
    for r in range(1, 3):
        for c in range(5):
            axp = fig.add_subplot(gs[r, c], sharey=profile_axes[0] if profile_axes else None)
            profile_axes.append(axp)
    ax_cbar_slot = fig.add_subplot(gs[0, 5]); pos = ax_cbar_slot.get_position(); ax_cbar_slot.remove()
    ax_cbar = fig.add_axes([pos.x0 + pos.width*0.75*0.76, pos.y0 + pos.height*0.005,
                            pos.width*0.24, pos.height*0.99])
    ax_map = fig.add_subplot(gs[1:, 5])

    # ---- curtain ----
    try:
        cs = fv.plot_curtain(polyline, var, time=model_date, ax=ax_transect,
                             ec='face', clim=clim, colorbar=False, cmap=cmap)
        try:
            cbar = fig.colorbar(cs, cax=ax_cbar, orientation='vertical')
        except Exception:
            sm = cm.ScalarMappable(norm=colors.Normalize(*clim), cmap=cmap); sm.set_array([])
            cbar = fig.colorbar(sm, cax=ax_cbar, orientation='vertical')
        cbar.set_label(clabel, fontsize=9); cbar.ax.tick_params(labelsize=8)
        y_top = max(ax_transect.get_ylim())
        for name in station_points:
            x_ch = manual_station_chainage.get(name)
            if x_ch is None:
                continue
            ax_transect.axvline(x_ch, color='white', ls='--', lw=1.0, alpha=0.9, zorder=8)
            ax_transect.plot(x_ch, y_top, marker='v', ms=6, mfc='white', mec='black',
                             clip_on=False, zorder=9)
            ax_transect.text(x_ch + 120, y_top - 1.1, name, fontsize=8, color='black',
                             weight='bold', zorder=10)
        ax_transect.set_xlabel('Chainage (m)', fontsize=10)
        ax_transect.set_ylabel('Depth (m)', fontsize=10)
        ax_transect.set_title(f'{var} transect | {pd.Timestamp(model_date).strftime("%Y-%m-%d %H:%M")}',
                              fontsize=11, fontweight='bold', pad=8)
        ax_transect.text(0.02, 1.05, 'N', transform=ax_transect.transAxes, fontsize=12, fontweight='bold', va='bottom')
        ax_transect.text(0.98, 1.05, 'S', transform=ax_transect.transAxes, fontsize=12, fontweight='bold', va='bottom', ha='right')
    except Exception as e:
        ax_transect.text(0.5, 0.5, f'Curtain error: {e}', transform=ax_transect.transAxes, ha='center', va='center')
        ax_cbar.set_axis_off()

    # ---- model profiles at each station ----
    for idx, site in enumerate(sites):
        ax = profile_axes[idx]
        try:
            prof = fv.get_profile(station_points[site], variables=[var], time=model_date)
            if prof is not None:
                prof_at_t = prof.sel(Time=model_date, method='nearest') if 'Time' in prof.dims else prof
                x_mod = np.asarray(prof_at_t[var]).ravel()
                z_mod = np.asarray(prof_at_t['Z']).ravel()
                ok = np.isfinite(x_mod) & np.isfinite(z_mod)
                if ok.sum() > 0:
                    ax.plot(x_mod[ok], z_mod[ok], '-s', color='steelblue', alpha=0.8,
                            lw=2, ms=3, label='Model')
        except Exception:
            pass
        ax.set_title(site, fontweight='bold', fontsize=10)
        ax.set_xlim(*clim); ax.set_ylim(y_min, y_max)
        ax.set_yticks(np.arange(-22, 2, 2)); ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', labelrotation=30)
        if idx < 5:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel(clabel, fontsize=9)
        if idx in (0, 5):
            ax.set_ylabel('Depth (m)', fontsize=9)
        else:
            ax.tick_params(labelleft=False)

    # ---- map inset ----
    try:
        fv.plot(var, time=model_date, ax=ax_map, datum='depth', cmap=cmap, clim=clim,
                shading='interp', colorbar=False, boundary=True)
        ax_map.plot(polyline[:, 0], polyline[:, 1], 'r-', lw=2)
        for name, (lon, lat) in station_points.items():
            ax_map.plot(lon, lat, 'o', ms=5, mfc='yellow', mec='black', zorder=6)
            ax_map.text(lon + 0.0018, lat + 0.0018, name, fontsize=8, color='red', weight='bold', zorder=7)
        lon_min, lon_max = polyline[:, 0].min(), polyline[:, 0].max()
        lat_min, lat_max = polyline[:, 1].min(), polyline[:, 1].max()
        lon_pad = max((lon_max - lon_min) * 0.08, 0.002); lat_pad = max((lat_max - lat_min) * 0.08, 0.002)
        ax_map.set_xlim(lon_min - lon_pad, lon_max + lon_pad)
        ax_map.set_ylim(lat_min - lat_pad, lat_max + lat_pad)
        ax_map.set_aspect('equal', adjustable='box')
        ax_map.grid(True, color='darkgrey', alpha=0.7, lw=0.5)
        ax_map.set_xticks(np.linspace(lon_min - lon_pad, lon_max + lon_pad, 3))
        ax_map.yaxis.set_major_locator(mticker.MaxNLocator(4))
        ax_map.tick_params(axis='y', labelrotation=90)
        ax_map.set_title('Transect Map', fontsize=10, fontweight='bold')
        ax_map.set_xlabel('Longitude', fontsize=8)
    except Exception as e:
        ax_map.text(0.5, 0.5, f'Map error: {e}', transform=ax_map.transAxes, ha='center', va='center')
        ax_map.set_axis_off()

    fig.legend(handles=[Line2D([0], [0], color='steelblue', lw=2, marker='s', ms=5, label='Model')],
               loc='lower right', fontsize=9, frameon=True, bbox_to_anchor=(0.69, 0.1))
    fig.subplots_adjust(top=0.90, bottom=0.08, left=0.06, right=0.99)
    if save:
        fname = OUT_DIR / f'transect_{var}_{pd.Timestamp(model_date).strftime("%Y%m%d_%H%M")}.png'
        fig.savefig(fname, dpi=200, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fname if save else None

In [ ]:
# === Generate figures ========================================================
dates_to_plot = snapshot_dates[:1] if TEST_MODE else snapshot_dates
print(f'{"TEST" if TEST_MODE else "FULL"} run: {len(dates_to_plot)} date(s) x {len(VARIABLES)} vars '
      f'= {len(dates_to_plot)*len(VARIABLES)} figures')
for var in VARIABLES:
    for d in dates_to_plot:
        out = make_transect_figure(var, d, save=True, show=TEST_MODE)
        print(f'  saved {out.name}')
print('Done.')